In [ ]:
# Colab setup
import sys
if "google.colab" in sys.modules:
    %pip install -q "pyedautils>=0.0.32" ephem statsmodels plotly ipywidgets

import plotly.io as pio
pio.renderers.default = "notebook_connected"

# Mollier h-x Diagram

```{admonition} Goal
:class: tip
Display measured indoor conditions on a Mollier h-x (psychrometric) diagram — the relationship between temperature, humidity and enthalpy of moist air, with a comfort zone, dashed warning bands, the outside air with its ventilation line and frequency contour lines.
```

```{admonition} Data Basis
:class: note
We use the bundled dataset `room_climate_sample.csv` shipped with **pyedautils** —
anonymised hourly temperature and humidity from a real sensor in a cool,
humidity-controlled room. It is loaded directly from the installed package
(`importlib.resources`), so no download is needed.
```

```{admonition} Reading the diagram
:class: seealso
New to the h-x diagram? The pyedautils guide [The h,x diagram in five minutes](https://retomarek.github.io/pyedautils/guides/mollier_hx.html) explains
[how to read it](https://retomarek.github.io/pyedautils/guides/mollier_hx.html#hx-reading-the-diagram), [the dew point](https://retomarek.github.io/pyedautils/guides/mollier_hx.html#hx-dew-point),
[the comfort zone](https://retomarek.github.io/pyedautils/guides/mollier_hx.html#hx-comfort) and [how to save energy](https://retomarek.github.io/pyedautils/guides/mollier_hx.html#hx-saving-energy).
```

## Data Preparation

In [ ]:
import pandas as pd
from importlib import resources

data_path = resources.files("pyedautils") / "data" / "room_climate_sample.csv"
df = pd.read_csv(data_path)   # columns: timestamp, humidity, temperature
df.head()

## Visualization

Pass the DataFrame to `plot_mollier_hx`. Measured points are coloured by season;
the default **comfort zone** (T 20–26 °C, φ 30–65 %) is drawn as the green polygon.
All charts are interactive — hover a point for details.

Every diagram below is drawn for the building's **site altitude** (here 450 m a.s.l.):
the matching air pressure comes from `pressure_from_altitude(...)`, so the psychrometric
relations are correct for the thinner air. Altitude and pressure are written in the
top-right corner.

In [ ]:
from pyedautils.plots import plot_mollier_hx
from pyedautils._mollier import pressure_from_altitude
from IPython.display import HTML, display

# Site altitude — the air pressure follows from it, so the psychrometric relations
# match the thinner air. Defined once and reused by every chart below.
altitude = 450  # m a.s.l.
pressure = pressure_from_altitude(altitude)

html = plot_mollier_hx(df, altitude=altitude, pressure=pressure, height=700)
display(HTML(html))

## Comfort zone & warning bands

Two **dashed warning bands** around the green comfort zone show how far a point is off:
`comfort_zone_orange` and `comfort_zone_red`, each given as **± offsets** around the comfort
zone (`temperature` in K, `rel_humidity` as a fraction, `abs_humidity` in kg/kg).
`comfort_warning_bands()` derives these offsets from the severity stages of the
[Comfort Compass](comfortCompass.ipynb), so both charts grade a room alike.

The `abs_humidity` limit (here 11.5 g/kg) closes the zones on the right: air above it is
flagged even when its relative humidity still looks fine.

In [ ]:
from pyedautils.comfort import comfort_warning_bands

# Mild / moderate stage limits: K, %-points and kg/kg off the comfort zone
bands = comfort_warning_bands(temp_stage_limits=(1.0, 2.5),
                              hum_stage_limits=(5.0, 10.0),
                              hum_abs_stage_limits=(0.001, 0.0025))

html = plot_mollier_hx(
    df,
    altitude=altitude, pressure=pressure,
    comfort_zone={"temperature": (20, 26), "rel_humidity": (0.30, 0.65),
                  "abs_humidity": (0, 0.0115)},
    comfort_label="Comfort zone (≤ 11.5 g/kg)",
    comfort_zone_orange={**bands["orange"], "label": "± 1 K / 5 % / 1 g/kg"},
    comfort_zone_red={**bands["red"], "label": "± 2.5 K / 10 % / 2.5 g/kg"},
    height=700,
)
display(HTML(html))

## Outside air & ventilation

Does airing make the room drier? Pass the outside air as `outdoor=(°C, % rH)`: it is drawn
as a blue circle, joined to the latest room reading by a dashed **ventilation line** along
which the room air moves when you air it. Only the **absolute** humidity decides: this cold,
damp night air (2 °C, 85 %) holds less water than the room air, so airing **dries** the
room. In a cool cellar on a muggy summer day it is the other way round
([examples in the guide](https://retomarek.github.io/pyedautils/guides/mollier_hx.html#hx-saving-energy)).

In [ ]:
from pyedautils.comfort import ventilation_effect

outside = (2, 85)                                   # deg C, % rH
room = tuple(df.iloc[-1][["temperature", "humidity"]])
print("Airing", ventilation_effect(room, outside, pressure=pressure))

html = plot_mollier_hx(df, altitude=altitude, pressure=pressure,
                       outdoor=outside, comfort_zone_orange=True, height=700)
display(HTML(html))

## Frequency contour lines

With enough data, `show_frequency=True` overlays cumulative-frequency contour lines:
each line labelled *N* encloses all but *N* units per year, so the outer lines mark
the rare extremes and the inner lines the typical core. Use `frequency_unit="days"`
to count distinct calendar days instead of hours.

Here we switch to a full **outdoor weather reference year for Zürich-Kloten** (bundled
with pyedautils) — the annual spread from cold, dry winters to warm, humid summers is
exactly what the contours are meant to show. The comfort zone is hidden
(`comfort_zone=False`), as outdoor weather is not about indoor comfort.

In [ ]:
df_kloten = pd.read_csv(
    resources.files("pyedautils") / "data" / "kloten_weather_sample.csv")

html = plot_mollier_hx(df_kloten, altitude=altitude, pressure=pressure,
                       comfort_zone=False,
                       show_frequency=True, frequency_unit="hours", height=700)
display(HTML(html))

## Customisation

::::{dropdown} Axis ranges, altitude, curve families, texts & convention
- `domain_x` / `domain_y` zoom into a region — absolute humidity [kg/kg] and the
  y-coordinate (≈ temperature at x = 0).
- `altitude` / `pressure` set the site altitude (m a.s.l., shown top-right); pass
  `pressure_from_altitude(...)` so the psychrometric relations match the thinner air.
- `show_temperature`, `show_density`, `show_rel_humidity`, `show_enthalpy`,
  `show_abs_humidity` toggle each iso-line family individually.
- `convention` switches the coordinate normalisation between `'classical'`
  (default, Recknagel) and `'glueck'`.
- `height` sets the diagram height in pixels.
- `captions` and `texts` replace any visible text, e.g. to translate the chart.

```python
from pyedautils._mollier import pressure_from_altitude

html = plot_mollier_hx(
    df,
    altitude=1500,                            # site altitude, shown top-right
    pressure=pressure_from_altitude(1500),    # matching air pressure
    domain_x=(0.002, 0.014),   # abs. humidity 2–14 g/kg
    domain_y=(10, 35),         # y-range ≈ 10–35 °C
    show_density=False,
    convention="classical",
    height=700,
)
```
::::